# Analyse des Dépendances Syntaxiques avec spaCy
Analyse des reviews par langue avec détection et étude des structures syntaxiques

## 1. Installation des dépendances

Dans cette section, les bibliothèques Python nécessaires au pipeline sont installées automatiquement (`pandas`, `spaCy`, `langdetect`, etc.).
L’objectif est d’assurer que l’environnement est prêt avant toute étape d’analyse.

In [2]:
# Installation des packages nécessaires (versions GPU-compatibles)
import subprocess
import sys

packages = [
    'numpy==2.2.6',
    'pandas==2.3.1',
    'scipy==1.15.3',
    'cupy-cuda12x==13.3.0',
    'spacy==3.8.7',
    'thinc==8.3.6',
    'langdetect',
    'matplotlib',
    'seaborn',
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

print('Packages installés avec succès!')

Packages installés avec succès!


## 2. Téléchargement des modèles spaCy

Cette section télécharge les modèles spaCy multilingues utilisés ensuite pour l’annotation syntaxique.
Chaque modèle correspond à une langue ciblée dans le corpus.

In [ ]:
import subprocess
import sys

# Installer directement les modèles spaCy compatibles avec spacy 3.8.x
model_packages = [
    'fr-core-news-sm==3.8.0',
    'en-core-web-sm==3.8.0',
    'es-core-news-sm==3.8.0',
    'de-core-news-sm==3.8.0',
    'pt-core-news-sm==3.8.0',
    'it-core-news-sm==3.8.0',
]

for model_pkg in model_packages:
    print(f'Installation de {model_pkg}...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', model_pkg, '-q'])
        print(f'  ✓ {model_pkg} installé')
    except Exception:
        print(f'  ✗ Erreur lors de l\'installation de {model_pkg}')

print('\nInstallations des modèles terminées!')

Installation de fr-core-news-sm==3.8.0...
  ✓ fr-core-news-sm==3.8.0 installé
Installation de en-core-web-sm==3.8.0...
  ✓ en-core-web-sm==3.8.0 installé
Installation de es-core-news-sm==3.8.0...
  ✓ es-core-news-sm==3.8.0 installé
Installation de de-core-news-sm==3.8.0...
  ✓ de-core-news-sm==3.8.0 installé
Installation de pt-core-news-sm==3.8.0...
  ✓ pt-core-news-sm==3.8.0 installé
Installation de it-core-news-sm==3.8.0...


## 3. Imports et chargement des données

Ici, le notebook importe les modules principaux, initialise les paramètres globaux (temps, énergie, CO2, taille d’échantillon) et charge le jeu de données source.
Cette étape pose le cadre de calcul pour toutes les sections suivantes.

In [5]:
import pandas as pd
import numpy as np
import spacy
from langdetect import detect, detect_langs
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from spacy import displacy
import warnings
import time
warnings.filterwarnings('ignore')

# Paramètres globaux temps/carbone (un seul compteur pour tout le script)
POWER_WATTS = 150  # puissance machine
FR_GRID_KGCO2_PER_KWH = 0.056  # électricité française (kgCO2e/kWh)
SCRIPT_START = time.perf_counter()

# Taille d'échantillon globale
# - valeur entière: nombre de lignes traitées
# - "total": traiter tout le fichier filtré
Sample_Size = 20000
k = Sample_Size

# Configuration des visualisations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Charger les données
df = pd.read_csv('data/reviews_select.csv', index_col=0)
print(f"Données chargées: {len(df)} reviews")
print(f"\nColonnes: {df.columns.tolist()}")
print(f"\nAperçu:")
df.head()

Données chargées: 530818 reviews

Colonnes: ['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'langue', 'langue_short', 'nwords']

Aperçu:


,listing_id,id,date,reviewer_id,reviewer_name,comments,langue,langue_short,nwords
1,5396.0,9.164898e+17,2023-06-18,497745629.0,Grazia,"Alloggio confortevole e pratico, dotato di tut...",it,Latin,18.0
2,5396.0,9.216001e+17,2023-06-25,70206366.0,Benjamin,Très bon emplacement pour cet appartement typi...,fr,French,27.0
3,5396.0,9.266632e+17,2023-07-02,41320355.0,Julia,"What a wonderful gem. Great location, it was ...",en,English,28.0
4,5396.0,9.288200e+17,2023-07-05,259277676.0,GLori,We had a lovely 3 night stay. Everything was ...,en,English,31.0
5,5396.0,9.295177e+17,2023-07-06,93503104.0,James,Great location. Very calm and quiet. Small but...,en,English,31.0


In [21]:
# Forcer le traitement sur l'ensemble du dataset
Sample_Size = 'total'
k = Sample_Size
print('Sample_Size forcé à:', Sample_Size)

Sample_Size forcé à: total


## 4. Analyse de la distribution des langues

Cette partie fournit un diagnostic rapide de la distribution linguistique du corpus (codes courts et libellés de langue).
Elle permet de vérifier la couverture des langues avant l’analyse syntaxique.

In [7]:
# Résumé textuel (sans visualisation graphique)
print("Langue par code court:")
langue_dist = df['langue_short'].value_counts()
print(langue_dist)

print("\nLangue par nom:")
langue_dist_name = df['langue'].value_counts()
print(langue_dist_name.head(15))

print(f"\nTotal de {len(langue_dist)} langues différentes détectées")

Langue par code court:
langue_short
English    269778
French     129519
Latin       56071
Others      41429
German      22314
Name: count, dtype: int64

Langue par nom:
langue
en    269778
fr    129519
es     29932
de     22314
it     14528
pt     10204
ko      8554
nl      7538
zh      5339
ja      3601
ru      1929
da      1782
tr      1484
ca      1407
pl      1147
Name: count, dtype: int64

Total de 5 langues différentes détectées


## 5. Sélection des langues principales et chargement des modèles spaCy

La section définit le mapping langue → modèle spaCy, puis charge les modèles disponibles en mémoire.
Le résultat est un dictionnaire `nlp_models` réutilisé pour toutes les annotations.

In [11]:
# Définir les principales langues et leurs modèles spaCy
spacy_models = {
    'fr': 'fr_core_news_sm',
    'en': 'en_core_web_sm',
    'es': 'es_core_news_sm',
    'de': 'de_core_news_sm',
    'pt': 'pt_core_news_sm',
    'it': 'it_core_news_sm'
}

# Vérifier les modèles par chargement effectif.
available_spacy_models = {}
for lang_code, model_name in spacy_models.items():
    try:
        _ = spacy.load(model_name, disable=['ner', 'textcat'])
        available_spacy_models[lang_code] = model_name
        print(f"✓ Modèle disponible: {lang_code} ({model_name})")
    except Exception as e:
        print(f"✗ Modèle introuvable: {lang_code} ({model_name}) -> {type(e).__name__}: {e}")

if not available_spacy_models:
    raise RuntimeError("Aucun modèle spaCy disponible. Exécute d'abord la cellule de téléchargement.")

# Compatibilité avec les cellules suivantes: nlp_models contient langue -> nom de modèle.
nlp_models = available_spacy_models
print(f"\n{len(nlp_models)} modèles spaCy prêts pour l'annotation")

✓ Modèle disponible: fr (fr_core_news_sm)
✓ Modèle disponible: en (en_core_web_sm)
✓ Modèle disponible: es (es_core_news_sm)
✓ Modèle disponible: de (de_core_news_sm)
✓ Modèle disponible: pt (pt_core_news_sm)
✓ Modèle disponible: it (it_core_news_sm)

6 modèles spaCy prêts pour l'annotation


## 6. Analyse des dépendances syntaxiques par langue

Ici, les données sont filtrées (langues supportées, commentaires non vides), puis échantillonnées selon `Sample_Size`/`k`.
Cette étape construit `df_sample`, l’échantillon effectivement analysé.

In [22]:
# Filtrer les reviews pour les langues principales avec modèles spaCy
# IMPORTANT: on utilise la colonne `langue` (codes ISO: en, fr, es, de, pt, it)
# 1) Taille pilotée par Sample_Size (ou alias k)
# 2) Exclure les textes linguistiquement non identifiés
df_work = df.copy()
df_work['lang_code'] = df_work['langue'].astype(str).str.lower().str.strip()

# Conserver uniquement les langues prises en charge (textes identifiés)
df_filtered = df_work[df_work['lang_code'].isin(nlp_models.keys())].copy()

# Exclure textes vides / NA
mask_text_valid = df_filtered['comments'].notna() & (df_filtered['comments'].astype(str).str.strip() != '')
df_filtered = df_filtered[mask_text_valid].copy()

# Taille d'échantillon paramétrable
sample_param = k if 'k' in globals() else Sample_Size

if isinstance(sample_param, str) and sample_param.lower() == 'total':
    df_sample = df_filtered.copy().reset_index(drop=True)
    sample_label = 'total'
else:
    sample_n = int(sample_param)
    if sample_n <= 0:
        raise ValueError('Sample_Size doit être un entier > 0 ou "total"')
    df_sample = df_filtered.head(min(sample_n, len(df_filtered))).copy().reset_index(drop=True)
    sample_label = str(sample_n)

print(f"Reviews éligibles (langue identifiée + texte valide): {len(df_filtered)}")
print(f"\nSample_Size={sample_label} -> {len(df_sample)} textes traités")
print("\nDistribution des langues dans l'échantillon:")
print(df_sample['lang_code'].value_counts())

Reviews éligibles (langue identifiée + texte valide): 476269

Sample_Size=total -> 476269 textes traités

Distribution des langues dans l'échantillon:
lang_code
en    269778
fr    129519
es     29932
de     22314
it     14522
pt     10204
Name: count, dtype: int64


## 7. Extraction des relations de dépendance syntaxique

Cette section exécute l’annotation syntaxique par langue, extrait les relations de dépendance et agrège des statistiques POS/DEP.
Une barre de progression `tqdm` facilite le suivi du traitement.

In [14]:
# Forcer le mode 'spawn' pour éviter les erreurs CUDA avec ProcessPoolExecutor sous Linux/Jupyter
import multiprocessing as mp

mp.set_start_method('spawn', force=True)
print('Multiprocessing start method:', mp.get_start_method())

Multiprocessing start method: spawn


In [23]:
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import pandas as pd
import subprocess

MAX_GPU_WORKERS = 4
BATCH_SIZE = 256
TEXT_CHAR_LIMIT = 2000


def _detect_gpu_count():
    try:
        import torch
        n = int(torch.cuda.device_count())
        if n > 0:
            return n
    except Exception:
        pass

    try:
        out = subprocess.check_output(['nvidia-smi', '-L'], text=True)
        return sum(1 for line in out.splitlines() if line.strip().startswith('GPU '))
    except Exception:
        return 0


def _safe_review_id(review_idx, row):
    if 'id' in row and pd.notna(row['id']):
        try:
            return str(int(row['id']))
        except Exception:
            return str(row['id'])
    return str(review_idx)


def _process_language_task(task):
    import spacy
    import cupy as cp
    from collections import Counter

    lang_code = task['lang_code']
    model_name = task['model_name']
    gpu_slot = task['gpu_slot']

    gpu_enabled = False
    if gpu_slot is not None:
        try:
            cp.cuda.Device(gpu_slot).use()
            spacy.require_gpu(gpu_slot)
            gpu_enabled = True
        except Exception:
            spacy.require_cpu()
            gpu_enabled = False
    else:
        spacy.require_cpu()

    nlp = spacy.load(model_name, disable=['ner', 'textcat'])

    dep_counter = Counter()
    pos_counter = Counter()
    root_counter = Counter()
    all_deps_lang = []
    ann_rows = []

    docs = nlp.pipe(task['texts'], batch_size=task['batch_size'])

    sent_count = 0
    token_count = 0

    for review_idx, review_id, doc in zip(task['review_idx'], task['review_id'], docs):
        for token in doc:
            if token.dep_ != 'ROOT':
                dep = {
                    'child': token.text,
                    'child_pos': token.pos_,
                    'dep': token.dep_,
                    'parent': token.head.text,
                    'parent_pos': token.head.pos_
                }
                all_deps_lang.append(dep)
                dep_counter[token.dep_] += 1
                pos_counter[token.pos_] += 1
            else:
                root_counter[token.text] += 1

        for sent in doc.sents:
            sent_text = sent.text.strip()
            if not sent_text:
                continue

            sent_count += 1
            for token in sent:
                token_id = token.i - sent.start + 1
                head = 0 if token.dep_ == 'ROOT' else (token.head.i - sent.start + 1)

                ann_rows.append({
                    'id': review_id,
                    'review_index': review_idx,
                    'lang': lang_code,
                    'sent_id': sent_count,
                    'sent_text': sent_text,
                    'token_id': token_id,
                    'form': token.text,
                    'lemma': token.lemma_ if token.lemma_ else '_',
                    'upos': token.pos_ if token.pos_ else '_',
                    'xpos': token.tag_ if token.tag_ else '_',
                    'feats': '_',
                    'head': head,
                    'deprel': token.dep_ if token.dep_ else '_',
                    'deps': '_',
                    'misc': '_'
                })
                token_count += 1

    return {
        'lang_code': lang_code,
        'gpu_enabled': gpu_enabled,
        'gpu_slot': gpu_slot,
        'total_tokens': int(sum(pos_counter.values())),
        'pos_distribution': dict(pos_counter),
        'dep_distribution': dict(dep_counter),
        'all_dependencies': all_deps_lang,
        'root_distribution': dict(root_counter),
        'ann_rows': ann_rows,
        'sent_count': sent_count,
        'token_count': token_count
    }


# Préparer les tâches par langue
gpu_count = _detect_gpu_count()
num_gpu_workers = min(MAX_GPU_WORKERS, gpu_count)

if num_gpu_workers > 0:
    print(f"GPU détectés: {gpu_count} | workers GPU disponibles: {num_gpu_workers}")
else:
    print("Aucun GPU CUDA détecté: fallback CPU.")

tasks = []
for task_idx, (lang_code, model_name) in enumerate(sorted(nlp_models.items())):
    lang_reviews = df_sample[df_sample['lang_code'] == lang_code].copy()
    if lang_reviews.empty:
        continue

    texts = lang_reviews['comments'].astype(str).str.slice(0, TEXT_CHAR_LIMIT).tolist()
    review_idx = lang_reviews.index.tolist()
    review_id = [_safe_review_id(idx, row) for idx, row in lang_reviews.iterrows()]

    gpu_slot = (task_idx % num_gpu_workers) if num_gpu_workers > 0 else None

    tasks.append({
        'lang_code': lang_code,
        'model_name': model_name,
        'texts': texts,
        'review_idx': review_idx,
        'review_id': review_id,
        'batch_size': BATCH_SIZE,
        'gpu_slot': gpu_slot
    })

print(f"Tâches prêtes: {len(tasks)} langues")

lang_dep_stats = {}
all_dependencies = defaultdict(lambda: defaultdict(int))
root_verbs_by_lang = {}
all_ann_rows = []

print("\nExtraction robuste en cours (séquentielle par langue, GPU assigné)...")
for task in tqdm(tasks, desc='Langues traitées'):
    result = _process_language_task(task)
    lang_code = result['lang_code']

    pos_dist = Counter(result['pos_distribution'])
    dep_dist = Counter(result['dep_distribution'])

    lang_dep_stats[lang_code] = {
        'total_tokens': result['total_tokens'],
        'pos_distribution': pos_dist,
        'dep_distribution': dep_dist,
        'all_dependencies': result['all_dependencies']
    }

    for dep_name, dep_count in dep_dist.items():
        all_dependencies[lang_code][dep_name] += dep_count

    root_verbs_by_lang[lang_code] = Counter(result['root_distribution'])
    all_ann_rows.extend(result['ann_rows'])

    mode = 'GPU' if result['gpu_enabled'] else 'CPU'
    slot_txt = f"GPU {result['gpu_slot']}" if result['gpu_slot'] is not None else 'CPU'
    print(f"  ✓ {lang_code.upper()} terminé ({mode}, {slot_txt})")

ann_df = pd.DataFrame(all_ann_rows)

print("\nExtraction terminée!")
print(f"Total annotations token: {len(ann_df)}")

GPU détectés: 4 | workers GPU disponibles: 4
Tâches prêtes: 6 langues

Extraction robuste en cours (séquentielle par langue, GPU assigné)...


Langues traitées:  17%|█▋        | 1/6 [00:30<02:31, 30.21s/it]

  ✓ DE terminé (GPU, GPU 0)


Langues traitées:  33%|███▎      | 2/6 [09:23<21:43, 325.98s/it]

  ✓ EN terminé (GPU, GPU 1)


...

Langues traitées:  83%|████████▎ | 5/6 [13:25<02:08, 128.24s/it]

  ✓ IT terminé (GPU, GPU 0)


Langues traitées: 100%|██████████| 6/6 [13:42<00:00, 137.09s/it]

  ✓ PT terminé (GPU, GPU 1)



Extraction terminée!
Total annotations token: 24362677


## 8 Export des annotations en CSV (avec colonne id)

Cette étape exporte les annotations token par token vers un CSV dans `data/`.
Le format inclut l’identifiant de review (`id`), les informations de phrase et les champs syntaxiques (lemma, upos, head, deprel, etc.).

In [24]:
from pathlib import Path

out_csv = Path('data/results_reviews_select_sample_annotations.csv')
out_csv.parent.mkdir(parents=True, exist_ok=True)

if 'ann_df' not in globals() or ann_df.empty:
    raise RuntimeError(
        "ann_df est vide ou absent. Exécute d'abord la cellule 'Extraction des relations de dépendance syntaxique'."
    )

ann_df.to_csv(out_csv, index=False, encoding='utf-8')

sent_count = ann_df['sent_id'].nunique() if 'sent_id' in ann_df.columns else 0
token_count = len(ann_df)

print(f"Fichier CSV exporté: {out_csv}")
print(f"Lignes d'annotation: {len(ann_df)}")
print(f"Phrases exportées: {sent_count}")
print(f"Tokens exportés: {token_count}")

ann_df.head()

Fichier CSV exporté: data/results_reviews_select_sample_annotations.csv
Lignes d'annotation: 24362677
Phrases exportées: 1154785
Tokens exportés: 24362677


,id,review_index,lang,sent_id,sent_text,token_id,form,lemma,upos,xpos,feats,head,deprel,deps,misc
0,971607015737973632,12,de,1,Meine Freundin und ich hatten einen schönen Au...,1,Meine,mein,DET,PPOSAT,_,2,nk,_,_
1,971607015737973632,12,de,1,Meine Freundin und ich hatten einen schönen Au...,2,Freundin,Freundin,NOUN,NN,_,0,ROOT,_,_
2,971607015737973632,12,de,1,Meine Freundin und ich hatten einen schönen Au...,3,und,und,CCONJ,KON,_,2,cd,_,_
3,971607015737973632,12,de,1,Meine Freundin und ich hatten einen schönen Au...,4,ich,ich,PRON,PPER,_,5,sb,_,_
4,971607015737973632,12,de,1,Meine Freundin und ich hatten einen schönen Au...,5,hatten,haben,VERB,VAFIN,_,3,cj,_,_


In [25]:
# Résumé textuel des dépendances (sans graphique)
print("Top dépendances par langue:")
for lang_code, stats in sorted(lang_dep_stats.items()):
    top10 = stats['dep_distribution'].most_common(10)
    print(f"\n{lang_code.upper()}:")
    if not top10:
        print("  Aucune dépendance extraite")
        continue
    for dep_name, dep_count in top10:
        print(f"  - {dep_name}: {dep_count}")

Top dépendances par langue:

DE:
  - nk: 267814
  - mo: 192762
  - punct: 148281
  - sb: 101703
  - cj: 72048
  - cd: 49651
  - pd: 47962
  - oc: 41315
  - oa: 39863
  - mnr: 30111

EN:
  - punct: 1823488
  - det: 1303207
  - nsubj: 1276447
  - prep: 1245780
  - pobj: 1173135
  - advmod: 1129805
  - amod: 994329
  - conj: 848090
  - cc: 760531
  - dobj: 515264

ES:
  - punct: 156400
  - det: 155612
  - case: 124105
  - advmod: 104988
  - amod: 74314
  - conj: 70603
  - obj: 65655
  - nsubj: 65136
  - cc: 63806
  - nmod: 56865

FR:
  - punct: 586580
  - case: 519530
  - det: 442087
  - advmod: 417727
  - nmod: 290055
  - nsubj: 287869
  - amod: 280773
  - conj: 250880
  - cc: 206085
  - obj: 132399

IT:
  - punct: 89912
  - case: 89392
  - det: 66871
  - conj: 50862
  - advmod: 49220
  - amod: 48996
  - obl: 44057
  - nmod: 40912
  - cc: 39369
  - nsubj: 30621

PT:
  - punct: 71908
  - case: 53717
  - det: 45178
  - advmod: 37010
  - conj: 33616
  - nsubj: 28486
  - amod: 27260
  - cc: 

## 9. Synthèse textuelle des POS tags par langue

La section suivante affiche une synthèse textuelle des POS tags les plus fréquents par langue.
Elle offre une lecture rapide des catégories morpho-syntaxiques dominantes.

In [26]:
# Résumé textuel des POS tags (sans graphique)
print("Top POS tags par langue:")
for lang_code, stats in sorted(lang_dep_stats.items()):
    top10 = stats['pos_distribution'].most_common(10)
    print(f"\n{lang_code.upper()}:")
    if not top10:
        print("  Aucun POS tag extrait")
        continue
    for pos_name, pos_count in top10:
        print(f"  - {pos_name}: {pos_count}")

Top POS tags par langue:

DE:
  - ADV: 183841
  - NOUN: 174640
  - PUNCT: 147372
  - DET: 113198
  - ADP: 93976
  - PRON: 83055
  - VERB: 61380
  - ADJ: 54075
  - CCONJ: 50536
  - PROPN: 36793

EN:
  - NOUN: 2560121
  - PUNCT: 1792061
  - ADJ: 1686060
  - DET: 1335363
  - ADP: 1332663
  - PRON: 1178658
  - ADV: 1098411
  - VERB: 914921
  - CCONJ: 757559
  - PROPN: 550325

ES:
  - NOUN: 216089
  - PUNCT: 155453
  - ADP: 152024
  - DET: 147642
  - ADJ: 114036
  - ADV: 100895
  - VERB: 76286
  - PRON: 67495
  - AUX: 62914
  - CCONJ: 62853

FR:
  - NOUN: 774875
  - ADP: 601594
  - PUNCT: 576826
  - DET: 440779
  - ADV: 423629
  - ADJ: 397909
  - VERB: 260617
  - PRON: 260468
  - AUX: 239856
  - CCONJ: 204455

IT:
  - NOUN: 130201
  - ADP: 97542
  - PUNCT: 88525
  - DET: 73542
  - ADJ: 71787
  - ADV: 52725
  - VERB: 43159
  - AUX: 42905
  - CCONJ: 39181
  - PRON: 25725

PT:
  - NOUN: 91487
  - PUNCT: 71738
  - ADP: 54985
  - DET: 46427
  - ADV: 39569
  - ADJ: 38904
  - VERB: 36514
  - CCONJ

## 10. Tableau récapitulatif syntaxique par langue

Ici, un tableau de synthèse rassemble les indicateurs clés par langue (tokens, relations, types uniques, dominante POS/DEP).
Ce tableau sert de vue consolidée des résultats intermédiaires.

In [27]:
# Créer un tableau récapitulatif
summary_data = []

for lang_code, stats in lang_dep_stats.items():
    top_dep = stats['dep_distribution'].most_common(1)[0][0] if stats['dep_distribution'] else 'N/A'
    top_pos = stats['pos_distribution'].most_common(1)[0][0] if stats['pos_distribution'] else 'N/A'
    
    summary_data.append({
        'Langue': lang_code.upper(),
        'Tokens analysés': stats['total_tokens'],
        'Relations de dépendance': len(stats['all_dependencies']),
        'Types de dépendances uniques': len(stats['dep_distribution']),
        'POS tags uniques': len(stats['pos_distribution']),
        'Dépendance dominante': top_dep,
        'POS dominant': top_pos
    })

summary_df = pd.DataFrame(summary_data)
print("\nRésumé des statistiques syntaxiques par langue:")
print(summary_df.to_string(index=False))


Résumé des statistiques syntaxiques par langue:
Langue  Tokens analysés  Relations de dépendance  Types de dépendances uniques  POS tags uniques Dépendance dominante POS dominant
    DE          1063241                  1063241                            41                17                   nk          ADV
    EN         14588513                 14588513                            44                18                punct         NOUN
    ES          1237092                  1237092                            30                18                punct         NOUN
    FR          4435065                  4435065                            35                17                punct         NOUN
    IT           705068                   705068                            40                18                punct         NOUN
    PT           478031                   478031                            35                17                punct         NOUN


## 12. Analyse des ROOT par langue (texte)

Cette partie isole les tokens `ROOT` pour observer les verbes principaux les plus fréquents selon la langue.
L’objectif est de caractériser la structure prédicative dominante des commentaires.

In [28]:
# Analyser les verbes principaux (ROOT) à partir du calcul déjà effectué
print("\n" + "="*60)
print("ANALYSE DES VERBES PRINCIPAUX (ROOT) PAR LANGUE")
print("="*60)

if 'root_verbs_by_lang' not in globals():
    raise RuntimeError(
        "root_verbs_by_lang absent. Exécute d'abord la cellule 'Extraction des relations de dépendance syntaxique'."
    )

for lang_code in sorted(nlp_models.keys()):
    verb_freq = root_verbs_by_lang.get(lang_code, Counter())

    print(f"\n{lang_code.upper()} - Verbes principaux les plus fréquents:")
    if not verb_freq:
        print("  Aucun ROOT extrait")
        continue

    for verb, count in verb_freq.most_common(10):
        print(f"  '{verb}' : {count} fois")


ANALYSE DES VERBES PRINCIPAUX (ROOT) PAR LANGUE

DE - Verbes principaux les plus fréquents:
  'ist' : 17826 fois
  'war' : 14419 fois
  'hat' : 4668 fois
  'haben' : 4284 fois
  'waren' : 3284 fois
  'sind' : 2391 fois
  'hatten' : 2213 fois
  'Wohnung' : 2137 fois
  'Dank' : 1426 fois
  'liegt' : 1361 fois

EN - Verbes principaux les plus fréquents:
  'was' : 218852 fois
  'is' : 159922 fois
  'recommend' : 42016 fois
  'had' : 40528 fois
  'were' : 34361 fois
  'stay' : 29697 fois
  'location' : 28377 fois
  'are' : 23237 fois
  'place' : 21191 fois
  'apartment' : 20841 fois

ES - Verbes principaux les plus fréquents:
  'ubicación' : 2476 fois
  'lugar' : 1683 fois
  'amable' : 1655 fois
  'tiene' : 1325 fois
  'es' : 1223 fois
  'recomiendo' : 1187 fois
  'ubicado' : 1099 fois
  'bien' : 1081 fois
  'apartamento' : 1013 fois
  'excelente' : 937 fois

FR - Verbes principaux les plus fréquents:
  'recommande' : 23255 fois
  'Merci' : 13385 fois
  'Appartement' : 13037 fois
  'passé'

## 15. Conclusions et résultats clés

La conclusion produit un résumé global: couverture linguistique, tableaux POS/DEP dominants et bilan temps/énergie/CO2.
Cette section fournit la sortie finale interprétable du notebook.

In [29]:
from datetime import datetime

print("\n" + "="*70)
print("RÉSUMÉ FINAL DE L'ANALYSE DES DÉPENDANCES SYNTAXIQUES")
print("="*70)

print(f"\n1. COUVERTURE LINGUISTIQUE")
print(f"   - Total de reviews analysées: {len(df)}")
print(f"   - Reviews retenues pour annotation: {len(df_sample)}")
langs_upper = [lang.upper() for lang in sorted(nlp_models.keys())]
print(f"   - Langues analysées: {', '.join(langs_upper)}")

# Tableaux de fréquences relatives (base: lignes = langues)
pos_abs_df = pd.DataFrame(
    {lang.upper(): stats['pos_distribution'] for lang, stats in sorted(lang_dep_stats.items())}
).T.fillna(0)
if not pos_abs_df.empty:
    pos_abs_df = pos_abs_df.loc[:, pos_abs_df.sum(axis=0) > 0]
pos_rel_df = pos_abs_df.div(pos_abs_df.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)


dep_abs_df = pd.DataFrame(
    {lang.upper(): stats['dep_distribution'] for lang, stats in sorted(lang_dep_stats.items())}
).T.fillna(0)
if not dep_abs_df.empty:
    dep_abs_df = dep_abs_df.loc[:, dep_abs_df.sum(axis=0) > 0]
dep_rel_df = dep_abs_df.div(dep_abs_df.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

# Transposition demandée: lignes = POS/deprel, colonnes = langues
pos_rel_t = pos_rel_df.T
pos_rel_t.columns.name = "langue"

dep_rel_t = dep_rel_df.T
dep_rel_t.columns.name = "langue"

print("\n2. RÉPARTITION RELATIVE DES TOKENS PAR POS (LIGNES=POS, COLONNES=LANGUES)")
print(pos_rel_t.round(4).to_string())

print("\n3. RÉPARTITION RELATIVE DES DÉPENDANCES (LIGNES=DEPREL, COLONNES=LANGUES)")
print(dep_rel_t.round(4).to_string())

# Bilan global temps/carbone (à la toute fin)
elapsed_sec_total = time.perf_counter() - SCRIPT_START
energy_kwh_total = (POWER_WATTS / 1000) * (elapsed_sec_total / 3600)
energy_wh_total = energy_kwh_total * 1000
co2_kg_total = energy_kwh_total * FR_GRID_KGCO2_PER_KWH
co2_g_total = co2_kg_total * 1000

print("\n4. BILAN GLOBAL TEMPS ET CARBONE")
print(f"   - Temps total de traitement: {elapsed_sec_total:.2f} s")
print(f"   - Énergie estimée ({POWER_WATTS}W): {energy_wh_total:.2f} Wh")
print(f"   - Émissions CO2 estimées (mix FR): {co2_g_total:.2f} gCO2e")

# Date/heure de fin d'exécution affichée avant le temps de calcul final.
execution_end = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"\n{'='*70}")
print(f"DATE ET HEURE DE FIN D'EXÉCUTION: {execution_end}")
print(f"TEMPS DE CALCUL FINAL: {elapsed_sec_total:.2f} s")
print(f"ENERGIE FINALE: {energy_wh_total:.2f} Wh")
print(f"CO2 FINAL: {co2_g_total:.2f} gCO2e")


RÉSUMÉ FINAL DE L'ANALYSE DES DÉPENDANCES SYNTAXIQUES

1. COUVERTURE LINGUISTIQUE
   - Total de reviews analysées: 530818
   - Reviews retenues pour annotation: 476269
   - Langues analysées: DE, EN, ES, FR, IT, PT

2. RÉPARTITION RELATIVE DES TOKENS PAR POS (LIGNES=POS, COLONNES=LANGUES)
langue      DE      EN      ES      FR      IT      PT
DET     0.1065  0.0915  0.1193  0.0994  0.1043  0.0971
CCONJ   0.0475  0.0519  0.0508  0.0461  0.0556  0.0527
PRON    0.0781  0.0808  0.0546  0.0587  0.0365  0.0379
VERB    0.0577  0.0627  0.0617  0.0588  0.0612  0.0764
ADJ     0.0509  0.1156  0.0922  0.0897  0.1018  0.0814
NOUN    0.1643  0.1755  0.1747  0.1747  0.1847  0.1914
ADP     0.0884  0.0914  0.1229  0.1356  0.1383  0.1150
PUNCT   0.1386  0.1228  0.1257  0.1301  0.1256  0.1501
ADV     0.1729  0.0753  0.0816  0.0955  0.0748  0.0828
SCONJ   0.0080  0.0128  0.0182  0.0060  0.0082  0.0218
PART    0.0118  0.0243  0.0001  0.0000  0.0000  0.0000
AUX     0.0260  0.0370  0.0509  0.0541  0.0609  0

In [9]:
import spacy
import thinc
import cupy
from thinc.api import prefer_gpu
print('spacy', spacy.__version__)
print('thinc', thinc.__version__)
print('cupy', cupy.__version__)
print('prefer_gpu(0)', prefer_gpu(0))

spacy 3.8.7
thinc 8.3.6
cupy 13.3.0
prefer_gpu(0) True


In [30]:
# Test fonctionnel CUDA sur chaque GPU H100
import cupy as cp
import torch
from thinc.api import prefer_gpu

print('torch.cuda.is_available =', torch.cuda.is_available())
print('torch.cuda.device_count =', torch.cuda.device_count())
print('thinc.prefer_gpu(0) =', prefer_gpu(0))

results = []
for i in range(torch.cuda.device_count()):
    with cp.cuda.Device(i):
        x = cp.random.random((3000, 3000), dtype=cp.float32)
        y = cp.random.random((3000, 3000), dtype=cp.float32)
        z = x @ y
        checksum = float(cp.sum(z).get())
        free_b, total_b = cp.cuda.runtime.memGetInfo()
        results.append((i, checksum, (total_b - free_b) / 1024**2, total_b / 1024**2))

print('\nRésultats par GPU:')
for i, checksum, used_mib, total_mib in results:
    print(f'GPU {i}: checksum={checksum:.3e}, mem_used={used_mib:.1f} MiB / {total_mib:.1f} MiB')

torch.cuda.is_available = True
torch.cuda.device_count = 4
thinc.prefer_gpu(0) = True

Résultats par GPU:
GPU 0: checksum=6.750e+09, mem_used=1016.9 MiB / 95321.1 MiB
GPU 1: checksum=6.752e+09, mem_used=1358.9 MiB / 95321.1 MiB
GPU 2: checksum=6.749e+09, mem_used=1152.9 MiB / 95321.1 MiB
GPU 3: checksum=6.749e+09, mem_used=1118.9 MiB / 95321.1 MiB
